# 2. Feature Engineering

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "synthetic_citizens_raw_data.csv"

RAW_DATA_PATH

PosixPath('/Users/yz/Desktop/DS找工/citizens-bank-fraud-prediction/data/raw/synthetic_citizens_raw_data.csv')

In [3]:
df = pd.read_csv(RAW_DATA_PATH)
df.shape

(17765, 49)

## Common Pipeline

This common pipeline creates the shared dataset used by all later model-specific pipelines. It should handle only operations that are valid for every model family: leakage removal, ID/no-variance removal, date parsing, sentinel handling, and basic availability/missingness features. Model-specific steps such as one-hot encoding, imputation, log transforms, and scaling should happen later inside Pipeline A/B/C.

In [ ]:
TARGET_COL = "return_target"
DATE_COL = "deposit_dt"

LEAKAGE_COLS = ["RDI_DT", "RETURN_REASON"]
ID_COLS = ["masked_dep_acct_num", "masked_id"]
LOW_VALUE_COLS = ["channel"]
COMMON_DROP_COLS = LEAKAGE_COLS + ID_COLS + LOW_VALUE_COLS + [DATE_COL]

RELATIONSHIP_BALANCE_SENTINEL = -99_999_999

PREVTRAN_COLS = [f"prevtran{i}" for i in range(1, 11)]
PREVTRANDATE_COLS = [f"prevtrandate{i}" for i in range(1, 11)]
DRAWEE_COLS = ["drawee_sum", "drawee_cnt", "drawee_avg", "drawee_max", "drawee_min"]

COMMON_DROP_COLS

In [ ]:
def add_common_features(raw_df):
    """Create shared, model-agnostic features without fitting any transformer."""
    df_common = raw_df.copy()

    # Parse deposit date, create reusable date features, then drop the raw date column later.
    df_common[DATE_COL] = pd.to_datetime(df_common[DATE_COL], errors="coerce")
    df_common["fe_deposit_month"] = df_common[DATE_COL].dt.month
    df_common["fe_deposit_week"] = df_common[DATE_COL].dt.isocalendar().week.astype("float")
    df_common["fe_deposit_dayofweek"] = df_common[DATE_COL].dt.dayofweek
    df_common["fe_deposit_is_weekend"] = df_common["fe_deposit_dayofweek"].isin([5, 6]).astype(int)

    # relationship_balance has a sentinel-like extreme value. Preserve it as a flag, then replace it with 0.
    if "relationship_balance" in df_common.columns:
        sentinel_mask = df_common["relationship_balance"] == RELATIONSHIP_BALANCE_SENTINEL
        df_common["fe_relationship_balance_is_sentinel"] = sentinel_mask.astype(int)
        df_common.loc[sentinel_mask, "relationship_balance"] = 0

    # Missingness/availability features. These are usually meaningful for account history fields.
    existing_prevtran_cols = [col for col in PREVTRAN_COLS if col in df_common.columns]
    existing_prevtrandate_cols = [col for col in PREVTRANDATE_COLS if col in df_common.columns]
    existing_drawee_cols = [col for col in DRAWEE_COLS if col in df_common.columns]

    if existing_prevtran_cols:
        df_common["fe_prevtran_valid_count"] = df_common[existing_prevtran_cols].notna().sum(axis=1)
        df_common["fe_prevtran_missing_count"] = df_common[existing_prevtran_cols].isna().sum(axis=1)
        df_common["fe_is_first_deposit"] = df_common[existing_prevtran_cols].isna().all(axis=1).astype(int)

    if existing_prevtrandate_cols:
        df_common["fe_prevtrandate_valid_count"] = df_common[existing_prevtrandate_cols].notna().sum(axis=1)
        df_common["fe_prevtrandate_missing_count"] = df_common[existing_prevtrandate_cols].isna().sum(axis=1)

    if existing_drawee_cols:
        df_common["fe_drawee_missing_flag"] = df_common[existing_drawee_cols].isna().all(axis=1).astype(int)

    if "rdis" in df_common.columns:
        df_common["fe_rdis_missing_flag"] = df_common["rdis"].isna().astype(int)

    # Drop columns that should not enter modeling after their useful information has been extracted.
    cols_to_drop = [col for col in COMMON_DROP_COLS if col in df_common.columns]
    df_common = df_common.drop(columns=cols_to_drop)

    return df_common


def stratified_split(data, target_col=TARGET_COL, train_size=0.70, valid_size=0.15, random_state=42):
    """Randomly split data while preserving the target positive rate in each split."""
    if not 0 < train_size < 1 or not 0 <= valid_size < 1 or train_size + valid_size >= 1:
        raise ValueError("train_size and valid_size must be valid proportions that leave a test set.")

    test_size = 1 - train_size - valid_size
    temp_size = valid_size + test_size

    train_df, temp_df = train_test_split(
        data,
        train_size=train_size,
        random_state=random_state,
        stratify=data[target_col],
    )

    relative_valid_size = valid_size / temp_size
    valid_df, test_df = train_test_split(
        temp_df,
        train_size=relative_valid_size,
        random_state=random_state,
        stratify=temp_df[target_col],
    )

    return (
        train_df.reset_index(drop=True),
        valid_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


def split_xy(split_df, target_col=TARGET_COL):
    y = split_df[target_col].copy()
    X = split_df.drop(columns=[target_col], errors="ignore").copy()
    return X, y


In [ ]:
df_common = add_common_features(df)

print("Raw shape:", df.shape)
print("Common shape:", df_common.shape)
print("Dropped columns:", [col for col in COMMON_DROP_COLS if col in df.columns])

display(df_common.head())

In [ ]:
train_df, valid_df, test_df = stratified_split(df_common)
X_train, y_train = split_xy(train_df)
X_valid, y_valid = split_xy(valid_df)
X_test, y_test = split_xy(test_df)

overall_target_rate = df_common[TARGET_COL].mean()
split_summary = pd.DataFrame({
    "split": ["overall", "train", "valid", "test"],
    "n_rows": [len(df_common), len(train_df), len(valid_df), len(test_df)],
    "positive_count": [
        int(df_common[TARGET_COL].sum()),
        int(y_train.sum()),
        int(y_valid.sum()),
        int(y_test.sum()),
    ],
    "target_rate": [
        overall_target_rate,
        y_train.mean(),
        y_valid.mean(),
        y_test.mean(),
    ],
})
split_summary["target_rate_diff_from_overall"] = split_summary["target_rate"] - overall_target_rate

display(split_summary)
print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("X_test shape:", X_test.shape)


## Next Model-Specific Pipelines

After this common pipeline is stable, build the model-specific branches:

1. Pipeline A: Logistic Regression baseline - one-hot encoding, imputation, missing indicators, log transforms, scaling.
2. Pipeline B: sklearn tree baseline - categorical encoding, explicit missing handling, no scaling.
3. Pipeline C: boosting-tree main model - engineered ratio/history/date features, minimal imputation, no scaling.